<a href="https://colab.research.google.com/github/Tejakondepudi-77/TEJA-S-AID/blob/main/Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

np.set_printoptions(precision=5, suppress=True)

print("System ready!")

System ready!


In [2]:
# ============================================================
# ELECTRICAL NETWORK NATURAL MODES - FRONTEND
# ============================================================

# ---------- TITLE ----------
display(HTML("""
<div style="
    background: linear-gradient(135deg,#111827,#2563eb);
    padding: 25px;
    border-radius: 18px;
    color: white;
    text-align: center;
    margin-bottom: 20px;">

    <h1 style="margin:0;">⚡ Electrical Network Natural Modes</h1>
    <h3 style="margin:8px 0;">B.Tech Mathematics Mini Project - B9</h3>
    <p>Eigenvalues • Eigenvectors • Natural Frequencies • Resonance</p>
</div>
"""))


# ---------- INPUT BOX CREATOR ----------
def input_box(description, value):
    return widgets.FloatText(
        value=value,
        description=description,
        layout=widgets.Layout(width='230px'),
        style={'description_width': '80px'}
    )


# ---------- INDUCTANCE INPUTS ----------
display(HTML("<h3>🔌 Inductance Parameters</h3>"))

L1_box = input_box("L1 (µH):", 10)
L2_box = input_box("L2 (µH):", 10)
L3_box = input_box("L3 (µH):", 10)

M12_box = input_box("M12 (µH):", 2)
M23_box = input_box("M23 (µH):", 2)
M13_box = input_box("M13 (µH):", 0)

inductance_box = widgets.VBox([
    widgets.HBox([L1_box, L2_box, L3_box]),
    widgets.HBox([M12_box, M23_box, M13_box])
])

display(inductance_box)


# ---------- CAPACITANCE INPUTS ----------
display(HTML("<h3>🔋 Capacitance Parameters</h3>"))

C1_box = input_box("C1 (pF):", 100)
C2_box = input_box("C2 (pF):", 100)
C3_box = input_box("C3 (pF):", 100)

display(widgets.HBox([C1_box, C2_box, C3_box]))


# ---------- BUTTON ----------
calculate_button = widgets.Button(
    description="▶ Calculate Natural Modes",
    button_style='primary',
    layout=widgets.Layout(
        width='260px',
        height='45px'
    )
)

reset_button = widgets.Button(
    description="↺ Reset",
    button_style='warning',
    layout=widgets.Layout(
        width='120px',
        height='45px'
    )
)

display(
    widgets.HBox(
        [calculate_button, reset_button],
        layout=widgets.Layout(
            justify_content='center',
            margin='20px'
        )
    )
)


# ---------- OUTPUT AREA ----------
output = widgets.Output()

display(output)


# ============================================================
# CALCULATION FUNCTION
# ============================================================

def calculate_modes(button=None):

    with output:

        clear_output(wait=True)

        # -----------------------------
        # Read inputs
        # -----------------------------

        uH = 1e-6
        pF = 1e-12

        L1 = L1_box.value * uH
        L2 = L2_box.value * uH
        L3 = L3_box.value * uH

        M12 = M12_box.value * uH
        M23 = M23_box.value * uH
        M13 = M13_box.value * uH

        C1 = C1_box.value * pF
        C2 = C2_box.value * pF
        C3 = C3_box.value * pF


        # -----------------------------
        # L matrix
        # -----------------------------

        L = np.array([
            [L1, M12, M13],
            [M12, L2, M23],
            [M13, M23, L3]
        ], dtype=float)


        # -----------------------------
        # C matrix
        # -----------------------------

        C = np.diag([C1, C2, C3])


        # -----------------------------
        # Check physical validity
        # -----------------------------

        L_check = np.linalg.eigvalsh(L)

        if np.any(L_check <= 0):

            display(HTML("""
            <div style="
                background:#fee2e2;
                padding:15px;
                border-radius:10px;
                color:#991b1b;">
                ⚠️ Invalid inductance matrix.
                Please check the self and mutual inductances.
            </div>
            """))

            return


        # -----------------------------
        # Circuit matrix
        # -----------------------------

        C_inverse = np.linalg.inv(C)

        A = np.linalg.solve(L, C_inverse)


        # -----------------------------
        # Eigenvalues/eigenvectors
        # -----------------------------

        eigenvalues, eigenvectors = np.linalg.eig(A)

        eigenvalues = np.real(eigenvalues)
        eigenvectors = np.real(eigenvectors)

        order = np.argsort(eigenvalues)

        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]


        # -----------------------------
        # Natural frequencies
        # -----------------------------

        omega = np.sqrt(eigenvalues)

        frequency = omega / (2 * np.pi)


        # -----------------------------
        # Normalize eigenvectors
        # -----------------------------

        modes = eigenvectors.copy()

        for k in range(3):

            modes[:, k] /= np.max(np.abs(modes[:, k]))

            max_index = np.argmax(np.abs(modes[:, k]))

            if modes[max_index, k] < 0:
                modes[:, k] *= -1


        # ====================================================
        # RESULT HEADER
        # ====================================================

        display(HTML("""
        <div style="
            background:#ecfdf5;
            border-left:6px solid #10b981;
            padding:15px;
            border-radius:10px;
            margin-top:15px;">

            <h2>✅ Calculation Completed</h2>
            <p>The three natural modes of the coupled LC network
            have been calculated.</p>
        </div>
        """))


        # ====================================================
        # NATURAL FREQUENCY CARDS
        # ====================================================

        cards = ""

        for k in range(3):

            cards += f"""
            <div style="
                display:inline-block;
                width:28%;
                margin:1%;
                padding:20px;
                border-radius:15px;
                background:#eff6ff;
                border:1px solid #bfdbfe;
                text-align:center;">

                <h3>Mode {k+1}</h3>

                <h2>{frequency[k]/1000:.3f} kHz</h2>

                <p>ω = {omega[k]:.2f} rad/s</p>

                <p>λ = {eigenvalues[k]:.3e}</p>

            </div>
            """

        display(HTML(
            "<h2>⚡ Natural Frequencies</h2>" + cards
        ))


        # ====================================================
        # RESULTS TABLE
        # ====================================================

        results = pd.DataFrame({

            "Mode": [1, 2, 3],

            "Eigenvalue λ = ω²":
                eigenvalues,

            "Angular Frequency ω":
                omega,

            "Frequency (Hz)":
                frequency,

            "Frequency (kHz)":
                frequency / 1000
        })

        display(HTML("<h2>📊 Eigenvalue Results</h2>"))

        display(
            results.style
            .format({
                "Eigenvalue λ = ω²": "{:.3e}",
                "Angular Frequency ω": "{:.3e}",
                "Frequency (Hz)": "{:.2f}",
                "Frequency (kHz)": "{:.3f}"
            })
        )


        # ====================================================
        # EIGENVECTOR TABLE
        # ====================================================

        mode_table = pd.DataFrame({

            "Mode": [1, 2, 3],

            "Circuit 1": modes[0, :],

            "Circuit 2": modes[1, :],

            "Circuit 3": modes[2, :]
        })

        display(HTML(
            "<h2>🔄 Current Distribution / Eigenvectors</h2>"
        ))

        display(
            mode_table.style.format("{:.4f}")
        )


        # ====================================================
        # MATRIX DISPLAY
        # ====================================================

        display(HTML("<h2>🧮 Inductance Matrix L</h2>"))

        display(
            pd.DataFrame(
                L / 1e-6,
                columns=["L1", "L2", "L3"],
                index=["L1", "L2", "L3"]
            ).style.format("{:.3f} µH")
        )


        # ====================================================
        # MODE PLOTS
        # ====================================================

        for k in range(3):

            plt.figure(figsize=(7,4))

            plt.bar(
                ["Circuit 1", "Circuit 2", "Circuit 3"],
                modes[:, k]
            )

            plt.axhline(0, linewidth=1)

            plt.title(
                f"Mode {k+1} Current Distribution "
                f"({frequency[k]/1000:.2f} kHz)"
            )

            plt.ylabel("Normalized Current")
            plt.xlabel("Circuit")

            plt.grid(axis="y", alpha=0.3)

            plt.show()


        # ====================================================
        # RESONANCE SPECTRUM
        # ====================================================

        f_min = 0.7 * np.min(frequency)
        f_max = 1.3 * np.max(frequency)

        freq_sweep = np.linspace(
            f_min,
            f_max,
            3000
        )

        # Gaussian peaks around natural frequencies.
        # Used only to visualize the resonance locations.
        width = 0.015 * np.mean(frequency)

        response = np.zeros_like(freq_sweep)

        for f in frequency:

            response += np.exp(
                -((freq_sweep - f) / width) ** 2
            )

        response /= np.max(response)


        plt.figure(figsize=(10,5))

        plt.plot(
            freq_sweep / 1000,
            response,
            linewidth=2
        )

        for k, f in enumerate(frequency):

            plt.axvline(
                f / 1000,
                linestyle="--",
                alpha=0.6
            )

            plt.text(
                f / 1000,
                1.02,
                f"Mode {k+1}",
                ha="center"
            )

        plt.title(
            "⚡ Resonance Frequencies of Coupled LC Network"
        )

        plt.xlabel("Frequency (kHz)")
        plt.ylabel("Normalized Response")

        plt.grid(alpha=0.3)

        plt.show()


        # ====================================================
        # CONCLUSION
        # ====================================================

        display(HTML("""
        <div style="
            background:#f3f4f6;
            padding:20px;
            border-radius:15px;
            margin-top:20px;">

            <h2>🎯 Project Conclusion</h2>

            <p>
            The coupled LC network has three natural modes.
            </p>

            <p>
            <b>Eigenvalues</b> determine the natural frequencies.
            </p>

            <p>
            <b>Eigenvectors</b> determine the relative current
            distribution in each mode.
            </p>

            <p>
            At the natural frequencies, the corresponding modes
            can be strongly excited, producing resonance.
            </p>

        </div>
        """))


# ============================================================
# RESET FUNCTION
# ============================================================

def reset_inputs(button=None):

    L1_box.value = 10
    L2_box.value = 10
    L3_box.value = 10

    M12_box.value = 2
    M23_box.value = 2
    M13_box.value = 0

    C1_box.value = 100
    C2_box.value = 100
    C3_box.value = 100

    with output:
        clear_output()


# Connect buttons
calculate_button.on_click(calculate_modes)
reset_button.on_click(reset_inputs)

Output()

In [4]:
display(HTML("""
<div style="
    background:#111827;
    color:white;
    padding:25px;
    border-radius:15px;
    margin-top:20px;">

<h2>📘 Mathematical Explanation</h2>

<p>
The coupled LC network is represented using matrices.
</p>

<p>
<b>Eigenvalues → Natural frequencies</b>
</p>

<p>
<b>Eigenvectors → Current distribution</b>
</p>

<p>
For each mode:
</p>

<p style="font-size:18px;">
lambda = omega²
</p>

<p style="font-size:18px;">
f = sqrt(lambda) / (2π)
</p>

<p>
The three modes correspond to the three independent
natural oscillations of the coupled network.
</p>

</div>
"""))